# Lab03 — 01. EDA → Preprocessing → Feature engineering

Khảo sát Rotten Tomatoes và chuẩn bị đặc trưng cho phân loại cảm xúc nhị phân. Notebook mặc định **MODE="prepare"**, tạo artifact từ pinned official Hugging Face dataset, không huấn luyện mô hình. Chạy từ trên xuống. Mỗi hình được **hiển thị inline và lưu PNG** trong `outputs/runs/<run_id>/` để dùng làm report.

| Phase | Câu hỏi / quyết định | Module/file tham chiếu | Artifact |
|---|---|---|---|
| 1 | Môi trường và reproducibility | `src/notebook_runtime.py`, `config/frozen_trial.json` | config, environment, manifest |
| 2 | Full splits và cân bằng nhãn | `src/datasets/loader.py` | data_preview, class_distribution |
| 3 | Missing, duplicates và overlap | `src/data_preparation.py` | data_quality PNG/JSON |
| 4 | Words/tokens theo lớp | `src/data_preparation.py` | lengths_by_class, token_length_distribution |
| 5 | Cutoff và tín hiệu phủ định/tương phản | `src/data_preparation.py` | preprocessing statistics/diagnostics |
| 6 | Text → WordPiece → IDs → mask | `src/datasets/loader.py` | tokenization_examples, attention_mask |
| 7 | Sparse TF-IDF và contextual features | `src/data_preparation.py` | feature_summary, tfidf_features |
| 8 | Train-only augmentation | `src/datasets/augmentation.py` | counts, diagnostics, examples |
| 9 | Chốt data/feature contract | `config/frozen_trial.json`, notebook 02 | seed, split, preprocessing policy |

`MODE="view"` chỉ đọc các artifact đã chọn trong `outputs/latest_preparation.json` hoặc `outputs/latest_report.json`, sau khi đã chạy pipeline. Quy trình tham chiếu: `docs/eda.md`, `docs/feature.md`, `docs/dl_data.md`. Discussion và architecture trong description; phương pháp/bằng chứng trong walkthrough.

## 1. Setup

Dùng Python 3.12 và requirements đã pin. Kaggle tạo interpreter riêng, không thay các thư viện ML của kernel hiển thị. MODE=prepare thực hiện EDA/feature extraction trên CPU; không cần GPU. Mỗi phiên có session ID riêng để chọn đúng artifact của phiên đó.

In [ ]:
from pathlib import Path
import subprocess, sys, json
MODE = "prepare"
DEVICE = "auto"
REPO_URL = "https://github.com/lacthui06/Lab-DL.git"
REPO_REF = "lab03"
start = Path.cwd().resolve()
REPO = next((p for p in (start,*start.parents) if (p/"labs/lab03/config/frozen_trial.json").is_file()), None)
if REPO is None and Path("/kaggle/working").is_dir():
    REPO = Path("/kaggle/working/Lab-DL")
    if not REPO.exists():
        subprocess.run(["git","clone","--branch",REPO_REF,"--single-branch",REPO_URL,str(REPO)],check=True)
assert REPO is not None, "Mở notebook trong repository hoặc trên Kaggle."
profile = json.loads((REPO/"labs/lab03/config/frozen_trial.json").read_text())["exercise2"]
assert profile["num_train_epochs"] == 5 and profile.get("early_stopping", {}).get("enabled"), "Cập nhật source branch lab03 cho đúng profile của notebook trước khi chạy."
sys.path.insert(0,str(REPO))
from labs.lab03.src.notebook_runtime import NotebookSession
session = NotebookSession(REPO, MODE, DEVICE)
assert "LAB03_SESSION_ID" in session.env, "Source repository chưa đồng bộ với notebook; cập nhật branch lab03 và mở session Kaggle mới."
session.setup()
from IPython.display import display


## 2. Official data contract

Rotten Tomatoes là sentence-level binary sentiment tiếng Anh: **0=negative, 1=positive**. Giữ nguyên official splits **8.530 train / 1.066 validation / 1.066 test**, không subsample, không chia lại test. Validation dùng chọn checkpoint/threshold; test chỉ đo sau khi khóa policy. Bảng và ảnh dưới đây được tạo từ dữ liệu thật của lượt prepare.

In [ ]:
session.prepare()
preview = session.read(session.paths["data"]/"data_preview.json")
display(preview["split_sizes"])
session.show("data", ["class_distribution.png", "train_examples.png"])


## 3. EDA chất lượng và leakage

Đếm missing/nonstring, blank, invalid label, exact duplicate và normalized duplicate theo split. Chuẩn hóa audit chỉ lowercase, gộp khoảng trắng; **không thay text đưa vào mô hình**. Overlap là số chuỗi chuẩn hóa giao giữa split, không phải tỷ lệ mẫu bị loại. Các con số khác 0 phải được thảo luận; giữ benchmark split chính thức và công khai giới hạn thay vì lọc heldout theo kết quả mô hình.

In [ ]:
quality = session.read(session.paths["data"]/"data_quality.json")
display(session.table("data", "data_quality.json", "splits"))
display(quality["normalized_cross_split_overlap"])
session.show("data", ["data_quality.png"])


## 4. EDA độ dài theo lớp

Whitespace word count không đồng nhất WordPiece count. `[CLS]`/`[SEP]` cũng tính vào sequence. Histogram theo lớp giúp phát hiện length bias; không coi review dài/ngắn là nhãn sentiment. Train token distribution quyết định cutoff, không chọn theo test accuracy.

In [ ]:
display(preview["train_token_lengths"])
session.show("data", ["lengths_by_class.png", "token_length_distribution.png"])


## 5. Preprocessing: chọn cutoff và giữ tín hiệu ngôn ngữ

Giữ punctuation, stopwords mang nghĩa, đặc biệt `not`/`but`; loại chúng có thể đảo sentiment. Tokenizer pretrained thực hiện lowercase và WordPiece. Bảng dưới đo quantiles và tỷ lệ train bị cắt ở cutoff64/96/128/256. Chọn `max_length=128` làm trần an toàn, kiểm tra lại trên distribution thực tế.

Dynamic padding pad tới câu dài nhất trong batch; không bắt mọi câu dài128. GPU FP16 pad bội8 để thuận tiện tính toán. Không chọn cutoff theo test accuracy. Negation/contrast là heuristic hỗ trợ kiểm duyệt augmentation, không phải feature nhân quả hay nhãn.

In [ ]:
display(session.table("data", "preprocessing_statistics.json"))
display(session.table("data", "text_signals.json"))
session.show("data", ["preprocessing_diagnostics.png"])


## 6. Kiểm tra text → tokens → input IDs → attention mask

Dùng bốn câu **thật từ train**, hai mỗi lớp. Bảng PNG rút gọn 12 tokens/IDs để dễ đọc; JSON giữ đủ tensor. Attention mask `1` là token hợp lệ, `0` là padding; label giữ nguyên, không được đưa vào input token. Cùng tokenizer revision và max_length được dùng trong modeling.

In [ ]:
examples = session.table("data", "tokenization_examples.json")
display(examples[["index", "text", "label", "tokens", "input_ids", "attention_mask"]])
session.show("data", ["preprocessing_examples.png", "attention_mask.png"])


## 7. Feature engineering: TF-IDF unigram/bigram

TF-IDF là baseline có thể giải thích: unigram giữ sentiment word, bigram giữ cụm như `not good`; `sublinear_tf=True`, `min_df=2`, tối đa50.000 features hạn chế vocabulary nhiễu. Fit vocabulary/IDF **chỉ trên original train** cho EDA này. Biểu đồ là top mean TF-IDF theo lớp, không phải causal importance hay trọng số classifier.

Trong notebook modeling, baseline fit lại vocabulary trên **effective augmented train** chung với Transformer, chọn Logistic Regression `C∈{0.1,1,10}` bằng validation Macro-F1. Hai bước dùng cùng cấu hình vectorizer, không dùng chung fitted object. Sparse array footprint là bytes của data/indices/indptr; không thay thế RAM peak của tiến trình.

In [ ]:
features = session.read(session.paths["data"]/"feature_summary.json")
display({k:v for k,v in features.items() if k != "terms"})
display(session.table("data", "feature_summary.json", "terms"))
session.show("data", ["tfidf_features.png"])


## 8. Feature engineering cho Transformer và tăng mẫu

Transformer dùng pretrained embeddings và contextual attention, không nối trực tiếp TF-IDF vào classifier head. Tăng diversity bằng thay một từ/cụm cùng cực tính, loại câu có phủ định/tương phản, chặn normalized duplicates và cân bằng số variants hai lớp. Nhãn synthetic kế thừa parent, cần kiểm duyệt nghĩa.

`max_ratio=0.25` là **giới hạn trên**, không ép đủ số lượng bằng sao chép. Tất cả original train được giữ; validation/test không tăng mẫu. Số thêm thực tế được tính và hiển thị bên dưới. Dropout, weight decay và early stopping ở modeling bổ sung regularization, không bảo đảm loại hết overfitting.

In [ ]:
augmentation = session.read(session.paths["augmentation"]/"augmentation.json")
display({k:augmentation[k] for k in ["original_train_size","added_count","effective_train_size","added_per_class"]})
session.show("augmentation", ["augmentation_counts.png", "augmentation_diagnostics.png"])


### 8.1. Kiểm duyệt ví dụ trước/sau

Đọc parent/variant, nhãn, từ thay và parent index. Câu khác ngữ cảnh có thể đổi nghĩa dù từ cùng cực tính. Chỉ kết luận chất lượng từ quan sát mẫu; không khẳng định mọi synthetic record đều đúng. Augmentation được tái tạo deterministic từ seed42, toàn bộ records và digest lưu trong run.

In [ ]:
display(session.table("augmentation", "augmentation.json", "records").head(12))
session.show("augmentation", ["augmentation_examples.png"])


## 9. Chốt data/feature contract trước modeling

| Thành phần | Quyết định | Vì sao |
|---|---|---|
| Splits | Official 8.530/1.066/1.066 | Không lấy test để tuning |
| Text | Giữ phủ định, punctuation; pretrained WordPiece | Giữ sentiment và tương thích pretrained |
| Sequence | max128, dynamic padding | Train distribution + tiết kiệm padding |
| Baseline | Sparse unigram/bigram TF-IDF + LR | Mốc so sánh đơn giản, fit train-only |
| Transformer | DistilBERT base, embedding +4/6 blocks frozen | Giảm số tham số học với ít dữ liệu |
| Augmentation | Same-polarity, dedup, tối đa25% | Thêm diversity có kiểm soát, không đổi heldout |
| Training mới | Max5 epoch, early stopping patience2 | Ngân sách ngắn, chọn min validation loss |

Mở [02_modeling.ipynb](02_modeling.ipynb), chạy từ trên xuống. Training tạo lại đúng preprocessing/augmentation từ config, không phụ thuộc biến Python của notebook01. Notebook01 không save model và không đo training accuracy.

In [ ]:
print("EDA artifacts:", session.paths["data"])
print("Augmentation artifacts:", session.paths["augmentation"])
print("Selection:", session.lab/"outputs/latest_preparation.json")
print("Mở notebook 02 để train; mỗi hình đã hiển thị cũng được lưu trong run folder.")


### Câu hỏi tự kiểm tra

1. Vì sao không bỏ `not` dù thường là stopword? Vì có thể đảo polarity.
2. TF-IDF có fit trên validation không? Không; validation chỉ chọn C/threshold trong modeling.
3. Tăng thêm25% có nghĩa phải đủ2.132 mẫu không? Không, là giới hạn; số thực tế phụ thuộc rule và dedup.
4. Có bảo đảm synthetic label đúng không? Không; weak labels cần kiểm duyệt và validation.
5. Mask có phải label không? Không; mask chỉ phân biệt token/padding.
6. Có cần rerun để lấy report không? Không, sau khi pipeline hoàn thành: hình, bảng, log và hash nằm trong outputs/.